In [1]:
import cv2
import requests
import time
import uuid

In [2]:
API_URL = "http://localhost:8000/api/v1/analysis/frame"

session_id = str(uuid.uuid4())

In [8]:
cap = cv2.VideoCapture(0)

print(f"Начало стриминга. Session ID: {session_id}")

last_level = "alert"
last_blink_rate = 0.0
last_perclos = 0.0
last_yawn_count = 0
last_events = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, encoded_image = cv2.imencode('.jpg', frame)
    if not success:
        continue

    files = {
        'frame': ('frame.jpg', encoded_image.tobytes(), 'image/jpeg')
    }
    data = {
        'session_id': session_id,
        'timestamp': time.time()
    }

    try:
        response = requests.post(API_URL, files=files, data=data)
        if response.status_code == 200:
            result = response.json()
            fatigue = result.get('fatigue_score', {})

            if fatigue:
                last_level = fatigue.get('level', 'alert')
                last_blink_rate = fatigue.get('blink_rate', 0.0)
                last_perclos = fatigue.get('perclos', 0.0)
                last_yawn_count = fatigue.get('yawn_count', 0)
                new_events = fatigue.get('events', [])

                if new_events:
                    last_events.extend(new_events)

    except Exception as e:
        pass
    
    current_time = time.time()
    last_events = [
        e for e in last_events 
        if current_time - e.get('timestamp_sec', current_time) < 10.0
    ]

    last_events = last_events[-3:]

    if last_level == "severe_fatigue":
        color = (0, 0, 255)       # Красный
        status_text = "DANGER: SEVERE FATIGUE"
    elif last_level == "mild_fatigue":
        color = (0, 255, 255)     # Желтый
        status_text = "WARNING: MILD FATIGUE"
    else:
        color = (0, 255, 0)       # Зеленый
        status_text = "STATUS: ALERT"

    cv2.putText(frame, status_text, (20, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
    
    cv2.putText(frame, f"Blinks/min: {last_blink_rate:.1f}", (20, 80), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
    
    perclos_pct = last_perclos * 100
    perclos_color = (0, 0, 255) if perclos_pct > 15 else (0, 255, 255) if perclos_pct > 8 else (255, 255, 255)
    cv2.putText(frame, f"PERCLOS: {perclos_pct:.1f}%", (20, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, perclos_color, 2, cv2.LINE_AA)
    
    yawn_color = (0, 0, 255) if last_yawn_count >= 4 else (0, 255, 255) if last_yawn_count >= 2 else (255, 255, 255)
    cv2.putText(frame, f"Yawns (5 min): {last_yawn_count}", (20, 145),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, yawn_color, 2, cv2.LINE_AA)

    y_offset = 185
    for event in last_events:  
        ev_type = event.get("event_type", "").replace("_", " ").upper()
        ev_dur = event.get("duration_sec")
        dur_text = f" ({ev_dur:.1f}s)" if ev_dur else ""
        cv2.putText(frame, f"! {ev_type}{dur_text}", (20, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 80, 255), 2, cv2.LINE_AA)
        y_offset += 35

    cv2.imshow("Driver Moniton", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Ресурсы освобождены.")

Начало стриминга. Session ID: a364f4f1-392f-4323-9c78-328acc15c9b3
Ресурсы освобождены.
